# 07 · Pitfalls and debugging

Each item: a tiny frame, the mistaken code and what it prints, one sentence, the fix and
what it prints. Read the outputs — the mistakes mostly run without error.

**What's in here**
(a) alignment · (b) assigning back from a filtered frame · (c) unsorted shift/rolling ·
(d) duplicated timestamps · (e) copy vs view · (f) chained comparison · (g) object dtype ·
(h) NaN semantics · (i) dtype drift · (j) `inplace` · (k) `df[mask]["x"] = 1` ·
(l) axis confusion · (m) merge duplication · (n) tz mixing · (o) leakage patterns ·
(p) day-first dates · (q) float equality · (r) removed APIs · (s) performance ·
audit checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## (a) Arithmetic aligns on the index — mismatched indexes give NaN, not an error

In [2]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
a + b

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64

`x` and `w` are NaN, and nothing warned you. The positional version *runs* and is silently wrong:

In [3]:
a.values + b.values

array([11, 22, 33])

Fix: decide what you mean. Align with fill, or reindex one to the other first.

In [4]:
a.add(b, fill_value=0)

w    30.0
x     1.0
y    12.0
z    23.0
dtype: float64

## (b) Assigning a Series computed on a filtered / sorted frame back to the original

`df` is in time order. We compute a rank on a **sorted-by-value** copy and assign it back.

In [5]:
df = pd.DataFrame({"v": [30, 10, 20]}, index=pd.to_datetime(["2023-01-01", "2023-01-02", "2023-01-03"]))
by_value = df.sort_values("v")
by_value["rank"] = [1, 2, 3]
by_value

,v,rank
2023-01-02,10,1
2023-01-03,20,2
2023-01-01,30,3


In [6]:
good = df.copy()
good["rank"] = by_value["rank"]           # Series: aligned by index -> each rank lands on its own date
good

,v,rank
2023-01-01,30,3
2023-01-02,10,1
2023-01-03,20,2


In [7]:
bad = df.copy()
bad["rank"] = by_value["rank"].values     # array: placed by position -> ranks in the wrong rows
bad

,v,rank
2023-01-01,30,1
2023-01-02,10,2
2023-01-03,20,3


01-01 has the largest value (30) so its rank should be 3. `good` says 3, `bad` says 1.
Assigning from a **subset** shows the same rule: NaN where the subset had no row.

In [8]:
sub = df[df["v"] > 15].copy()
sub["flag"] = 1
df2 = df.copy()
df2["flag"] = sub["flag"]        # aligned -> NaN for 01-02
df2

,v,flag
2023-01-01,30,1.0
2023-01-02,10,NaN
2023-01-03,20,1.0


## (c) Not sorting before `shift` / `rolling` / `merge_asof`

Rows are not in time order. `shift(1)` gives the *previous row*, which is not the previous hour.

In [9]:
u = pd.DataFrame({"time": pd.to_datetime(["2023-01-01 03:00", "2023-01-01 01:00", "2023-01-01 02:00"]), "v": [3, 1, 2]})
u["lag_wrong"] = u["v"].shift(1)
u

,time,v,lag_wrong
0,2023-01-01 03:00:00,3,NaN
1,2023-01-01 01:00:00,1,3.0
2,2023-01-01 02:00:00,2,1.0


In [10]:
u = u.sort_values("time").reset_index(drop=True)
u["lag_right"] = u["v"].shift(1)
u

,time,v,lag_wrong,lag_right
0,2023-01-01 01:00:00,1,3.0,NaN
1,2023-01-01 02:00:00,2,1.0,1.0
2,2023-01-01 03:00:00,3,NaN,2.0


After sorting, the lag at 02:00 is 1 (the 01:00 value), as it should be.

## (d) Duplicated timestamps

A timestamp appears twice. `shift(1)` is now off by one row after it, and `reindex` refuses.

In [11]:
d = pd.Series([1, 2, 2, 3], index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00", "2023-01-01 01:00", "2023-01-01 02:00"]))
print("unique:", d.index.is_unique)
pd.DataFrame({"d": d, "shift(1)": d.shift(1)})

unique: False


,d,shift(1)
2023-01-01 00:00:00,1,NaN
2023-01-01 01:00:00,2,1.0
2023-01-01 01:00:00,2,2.0
2023-01-01 02:00:00,3,2.0


The 02:00 row's "previous value" is the second copy of 01:00, not a real previous hour.

In [12]:
try:
    d.reindex(pd.date_range("2023-01-01 00:00", periods=3, freq="h"))
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot reindex on an axis with duplicate labels


In [13]:
d = d[~d.index.duplicated(keep="last")]
d

2023-01-01 00:00:00    1
2023-01-01 01:00:00    2
2023-01-01 02:00:00    3
dtype: int64

## (e) SettingWithCopyWarning — copy or view?

Filtering gives a frame that *may* be a view of the original. Writing into it is
unreliable: sometimes the original changes, sometimes not, and pandas may or may not
warn (`SettingWithCopyWarning`) depending on version and settings. The printed flag shows
whether it warned here.

In [14]:
import warnings
df = pd.DataFrame({"temp": [5.0, -999.0, 7.0]})
sub = df[df["temp"] < 0]
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    sub["temp"] = np.nan
    print("warned:", any("SettingWithCopy" in str(x.message) for x in w))
print(df)              # original: unchanged here, but you cannot rely on it

warned: False
    temp
0    5.0
1 -999.0
2    7.0


The two intended versions: change the original with `.loc`, or take an explicit copy.

In [15]:
df.loc[df["temp"] < 0, "temp"] = np.nan
df

,temp
0,5.0
1,NaN
2,7.0


In [16]:
sub = df[df["temp"].isna()].copy()   # a real copy: edit freely, original untouched
sub["temp"] = 0
print(sub)
print(df)

   temp
1     0
   temp
0   5.0
1   NaN
2   7.0


## (f) `a < x < b` does not work on a Series

Python evaluates it as `(a < x) and (x < b)`, and `and` on a Series is ambiguous.

In [17]:
x = pd.Series([1, 5, 9])
try:
    0 < x < 6
except ValueError as e:
    print("ValueError:", str(e)[:60])

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.boo


In [18]:
(x > 0) & (x < 6)

0     True
1     True
2    False
dtype: bool

## (g) Numbers stored as object dtype

One bad string turns the whole column into `object`. `mean()` fails; `sum()` may concatenate.

In [19]:
p = pd.Series(["1.5", "2.5", "missing"])
print(p.dtype)
try:
    p.mean()
except TypeError as e:
    print("TypeError:", str(e)[:60])

object
TypeError: Could not convert string '1.52.5missing' to numeric


In [20]:
pn = pd.to_numeric(p, errors="coerce")
print(pn)
print("mean:", pn.mean(), " bad values:", pn.isna().sum())

0    1.5
1    2.5
2    NaN
dtype: float64
mean: 2.0  bad values: 1


## (h) NaN semantics

In [21]:
n = pd.Series([1.0, np.nan, 3.0])
print("sum          :", n.sum())            # NaN skipped
print("mean         :", n.mean())           # NaN skipped -> 2.0, not 1.33
print("all-NaN sum  :", pd.Series([np.nan, np.nan]).sum())     # 0, not NaN
print("== np.nan    :", (n == np.nan).sum())                    # always 0
print("isna         :", n.isna().sum())

sum          : 4.0
mean         : 2.0
all-NaN sum  : 0.0
== np.nan    : 0
isna         : 1


`groupby` drops NaN keys by default, so group totals do not add up to the grand total.

In [22]:
g = pd.DataFrame({"tariff": ["Fixed", None, "TOU"], "kwh": [10, 20, 30]})
print(g.groupby("tariff")["kwh"].sum())
print("grand total:", g["kwh"].sum())

tariff
Fixed    10
TOU      30
Name: kwh, dtype: int64
grand total: 60


In [23]:
g.groupby("tariff", dropna=False)["kwh"].sum()

tariff
Fixed    10
TOU      30
NaN      20
Name: kwh, dtype: int64

## (i) Dtype drift: int → float, bool → object

Introducing a NaN (through reindex, merge, shift) changes the dtype. Integer ids become
floats like `100000.0`; booleans become `object`.

In [24]:
ints = pd.Series([1, 2, 3])
print(ints.dtype, "->", ints.shift(1).dtype)
print(ints.shift(1))

int64 -> float64
0    NaN
1    1.0
2    2.0
dtype: float64


In [25]:
flags = pd.Series([True, False])
print(flags.dtype, "->", flags.reindex([0, 1, 2]).dtype)
print("nullable alternative:", ints.astype("Int64").shift(1).dtype)

bool -> object
nullable alternative: Int64


## (j) `inplace=True` returns None

In [26]:
df = pd.DataFrame({"a": [1, 2]})
result = df.rename(columns={"a": "b"}, inplace=True)
print("result:", result)
print(df)

result: None
   b
0  1
1  2


Chaining after `inplace=True` fails because there is nothing to chain on. Prefer assignment.

In [27]:
df = pd.DataFrame({"a": [1, 2]})
df = df.rename(columns={"a": "b"})
df

,b
0,1
1,2


## (k) `df[mask]["x"] = 1` does nothing

`df[mask]` builds a temporary copy; the assignment lands on the copy and is thrown away.

In [28]:
df = pd.DataFrame({"v": [1, 2, 3], "x": [0, 0, 0]})
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df[df["v"] > 1]["x"] = 1
df

,v,x
0,1,0
1,2,0
2,3,0


In [29]:
df.loc[df["v"] > 1, "x"] = 1
df

,v,x
0,1,0
1,2,1
2,3,1


## (l) Axis confusion

`axis=0` runs **down** the rows (one result per column); `axis=1` runs **across** the columns (one result per row).

In [30]:
m = pd.DataFrame({"a": [1, 2], "b": [10, 20]}, index=["r1", "r2"])
print(m)
print()
print("sum(axis=0):"); print(m.sum(axis=0))
print()
print("sum(axis=1):"); print(m.sum(axis=1))

    a   b
r1  1  10
r2  2  20

sum(axis=0):
a     3
b    30
dtype: int64

sum(axis=1):
r1    11
r2    22
dtype: int64


`drop` follows the same rule: `axis=0` drops a row label, `axis=1` drops a column.

In [31]:
print(m.drop("r1", axis=0))
print()
print(m.drop("a", axis=1))

    a   b
r2  2  20

     b
r1  10
r2  20


## (m) Merge duplicating rows

Covered in detail in notebook 05. The two-line version:

In [32]:
left = pd.DataFrame({"k": ["A", "B"], "v": [1, 2]})
right = pd.DataFrame({"k": ["A", "A", "B"], "w": [10, 11, 20]})
m = pd.merge(left, right, on="k")
print("left", len(left), "-> merged", len(m))
m

left 2 -> merged 3


,k,v,w
0,A,1,10
1,A,1,11
2,B,2,20


## (n) tz-naive vs tz-aware comparisons

Comparing an aware timestamp with a naive one raises. Filtering with a string works because
pandas interprets the string in the index's zone.

In [33]:
t = pd.Series([1, 2], index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00"], utc=True))
try:
    t[t.index > pd.Timestamp("2023-01-01 00:30")]
except TypeError as e:
    print("TypeError:", str(e)[:60])

TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and Tim


In [34]:
t[t.index > pd.Timestamp("2023-01-01 00:30", tz="UTC")]

2023-01-01 01:00:00+00:00    2
dtype: int64

## (o) Look-ahead leakage patterns

**Interview check:** *"Your R² went from 0.71 to 0.93 after adding this feature. Why might
that be bad news?"* — Because the feature may contain the target. Five ways it happens.

1. Rolling without shift: the window includes the current row.

In [35]:
y = pd.Series([10, 20, 30, 40])
pd.DataFrame({"y": y, "leaks": y.rolling(2).mean(), "honest": y.shift(1).rolling(2).mean()})

,y,leaks,honest
0,10,NaN,NaN
1,20,15.0,NaN
2,30,25.0,15.0
3,40,35.0,25.0


2. Separate `dropna` on X and y, then `.values`: rows no longer match.

In [36]:
X = pd.DataFrame({"lag1": y.shift(1)})      # NaN in row 0
target = y.shift(-1)                          # NaN in row 3
Xd, yd = X.dropna(), target.dropna()
print("X rows:", Xd.index.tolist(), " y rows:", yd.index.tolist())

X rows: [1, 2, 3]  y rows: [0, 1, 2]


In [37]:
frame = pd.concat([X, target.rename("target")], axis=1).dropna()   # fix: one frame, one dropna
frame

,lag1,target
1,10.0,30.0
2,20.0,40.0


3. Standardising on the full sample: the test mean/std leak into training.

In [38]:
v = pd.Series([1, 2, 3, 100])          # last value is "test"
train = v.iloc[:3]
print("full-sample mean:", v.mean(), "  train-only mean:", train.mean())

full-sample mean: 26.5   train-only mean: 2.0


4. `bfill` / `interpolate` fill with the **future**; `ffill` does not.

In [39]:
g = pd.Series([1.0, np.nan, 3.0])
pd.DataFrame({"g": g, "ffill": g.ffill(), "bfill": g.bfill(), "interpolate": g.interpolate()})

,g,ffill,bfill,interpolate
0,1.0,1.0,1.0,1.0
1,NaN,1.0,3.0,2.0
2,3.0,3.0,3.0,3.0


5. `label="right"` stamps the bar with its **end** time: the 02:00 bar holds 00:00–01:00 data.

In [40]:
h = pd.Series([1, 2, 3, 4], index=pd.date_range("2023-01-01", periods=4, freq="h"))
pd.DataFrame({"label=left": h.resample("2h").sum(), "label=right": h.resample("2h", label="right").sum()})

,label=left,label=right
2023-01-01 00:00:00,3.0,NaN
2023-01-01 02:00:00,7.0,3.0
2023-01-01 04:00:00,NaN,7.0


## (p) `to_datetime` and day-first ambiguity

`03/04/2023` is 4 March or 3 April? pandas guesses month-first. Say `dayfirst=True` or give the `format`.

In [41]:
txt = pd.Series(["03/04/2023", "05/04/2023"])
print(pd.to_datetime(txt).dt.strftime("%Y-%m-%d").tolist(), "<- month first (default)")
print(pd.to_datetime(txt, dayfirst=True).dt.strftime("%Y-%m-%d").tolist(), "<- day first")
print(pd.to_datetime(txt, format="%d/%m/%Y").dt.strftime("%Y-%m-%d").tolist(), "<- explicit format")

['2023-03-04', '2023-05-04'] <- month first (default)
['2023-04-03', '2023-04-05'] <- day first
['2023-04-03', '2023-04-05'] <- explicit format


## (q) Float equality

In [42]:
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
f = pd.Series([0.1 + 0.2, 0.3])
print((f == 0.3).tolist(), np.isclose(f, 0.3).tolist())

False
True
[False, True] [True, True]


## (r) Removed in pandas 2.x (mention only)

`df.append` → `pd.concat([df, other])`. `Series.iteritems` → `items`. `df.ix` → `loc` / `iloc`.

## (s) Performance: `iterrows` vs vectorised; categoricals for memory

In [43]:
big = pd.DataFrame({"temp": np.random.default_rng(0).normal(10, 5, 50_000)})
%timeit [max(0, 15 - t) for t in big["temp"]]
%timeit (15 - big["temp"]).clip(lower=0)

9.35 ms ± 992 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


654 µs ± 122 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [44]:
region = pd.Series(["London"] * 50_000)
print("object  :", region.memory_usage(deep=True), "bytes")
print("category:", region.astype("category").memory_usage(deep=True), "bytes")

object  : 3150128 bytes
category: 50299 bytes


## 10-minute notebook audit checklist

1. What is one row? Is the key (timestamp, meter) unique?
2. Timestamps parsed, tz-aware, sorted?
3. Row counts printed before/after every merge and filter?
4. Any `object` column that should be numeric?
5. Any `.values` used where alignment mattered?
6. Every `rolling` shifted? Every `shift` in the right direction (`+` = past, `-` = future)?
7. Split chronological? Scaler / encoder fit on train only?
8. Any `bfill` / `interpolate` on features?
9. Result compared with a naive baseline?
10. A coefficient near 1 on a feature derived from the target?